# Second dataset: Sparkov simulated card transactions


In [ ]:
# ---------- Cell 1: configuration and imports ----------
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"; os.environ["OMP_NUM_THREADS"] = "2"   
import sys, time, json, platform, warnings, glob
try:
    import torch          # import torch before XGBoost / LightGBM
except Exception as _e:
    print("torch not available:", _e)
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

CFG = dict(
    DATA_DIR=os.environ.get("THESIS_SPARKOV_DIR", os.path.expanduser("~/Downloads")),
    SAMPLE_FRAC=float(os.environ.get("SPARKOV_FRAC", 0.25)),  
    SEEDS=[42, 43, 44],         
    GAT_SEEDS=[42, 43, 44],             
    RUN_GAT=os.environ.get("SPARKOV_GAT", "1") == "1",
    TREE_LR=0.1, MAX_TREES=1500, ES_ROUNDS=50,
    GAT_HIDDEN=32, GAT_HEADS=2, GAT_OUT=32, GAT_EPOCHS=150, GAT_LR=0.005, GAT_WD=5e-4,
    GAT_POSW=3.0, GAT_EVAL_EVERY=10, GAT_MAX_MIN=6,
    N_BOOT=200, SHAP_N=2000, OUT_DIR="sparkov_outputs",
)
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
SPLIT_FRAC, VAL_FRAC = 0.80, 0.10
print({k: v for k, v in CFG.items() if k != "DATA_DIR"})

{'SAMPLE_FRAC': 0.25, 'SEEDS': [42, 43, 44], 'GAT_SEEDS': [42, 43, 44], 'RUN_GAT': True, 'TREE_LR': 0.1, 'MAX_TREES': 1500, 'ES_ROUNDS': 50, 'GAT_HIDDEN': 32, 'GAT_HEADS': 2, 'GAT_OUT': 32, 'GAT_EPOCHS': 150, 'GAT_LR': 0.005, 'GAT_WD': 0.0005, 'GAT_POSW': 3.0, 'GAT_EVAL_EVERY': 10, 'GAT_MAX_MIN': 6, 'N_BOOT': 200, 'SHAP_N': 2000, 'OUT_DIR': 'sparkov_outputs'}


In [ ]:
# ---------- Cell 2: load, sample, order by time, split, tabular features ----------
def find_file(name):
    for d in [CFG["DATA_DIR"], os.getcwd(), os.path.expanduser("~/Desktop"), os.path.expanduser("~/Documents")]:
        hit = glob.glob(os.path.join(d, "**", name), recursive=True)
        if hit: return hit[0]
    raise FileNotFoundError(f"{name} not found. Put it in {CFG['DATA_DIR']} or set THESIS_SPARKOV_DIR.")

usecols = ["trans_date_trans_time","cc_num","merchant","category","amt","gender","state","zip","lat","long",
           "city_pop","job","dob","unix_time","merch_lat","merch_long","is_fraud"]
df = pd.concat([pd.read_csv(find_file("fraudTrain.csv"), usecols=usecols),
                pd.read_csv(find_file("fraudTest.csv"), usecols=usecols)], ignore_index=True)
print("full data:", df.shape, "fraud rate", round(df.is_fraud.mean(), 5))

if CFG["SAMPLE_FRAC"] < 1.0:
    df = df.sample(frac=CFG["SAMPLE_FRAC"], random_state=42)       
df["ts"] = pd.to_datetime(df["trans_date_trans_time"])
df = df.sort_values(["unix_time", "cc_num"], kind="mergesort").reset_index(drop=True)
n = len(df); split_idx = int(SPLIT_FRAC * n); val_start = int((1 - VAL_FRAC) * split_idx)
fit_sl, val_sl, test_sl = slice(0, val_start), slice(val_start, split_idx), slice(split_idx, n)
y = df["is_fraud"].values.astype(np.int8)
print(f"n={n:,}  fit={val_start:,}  val={split_idx-val_start:,}  test={n-split_idx:,}")
print("fraud rate  fit %.4f  val %.4f  test %.4f" % (y[fit_sl].mean(), y[val_sl].mean(), y[test_sl].mean()))

# derived columns
df["hour"] = df.ts.dt.hour; df["dow"] = df.ts.dt.dayofweek
df["age"] = (df.ts - pd.to_datetime(df.dob)).dt.days / 365.25
def haversine(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = np.sin((lat2-lat1)*p/2)**2 + np.cos(lat1*p)*np.cos(lat2*p)*np.sin((lon2-lon1)*p/2)**2
    return 12742 * np.arcsin(np.sqrt(a))
df["dist_km"] = haversine(df.lat, df.long, df.merch_lat, df.merch_long)
df["log_amt"] = np.log1p(df.amt)

cat_cols = ["category", "gender", "state", "job", "merchant", "cc_num"]
num_cols = ["amt","log_amt","city_pop","lat","long","merch_lat","merch_long","zip","age","hour","dow","dist_km"]
cat_codes = {}
for c in cat_cols:                                   
    book = {v: i for i, v in enumerate(pd.unique(df[c].iloc[:split_idx]))}
    cat_codes[c] = df[c].map(book).fillna(-1).astype(np.int32).values
from sklearn.preprocessing import StandardScaler
num = df[num_cols].fillna(0).values.astype(np.float32)
num = StandardScaler().fit(num[:split_idx]).transform(num).astype(np.float32)
X_tab = np.hstack([np.column_stack([cat_codes[c] for c in cat_cols]).astype(np.float32), num])
tab_names = cat_cols + num_cols
print("tabular features:", X_tab.shape[1])
spw = float((y[fit_sl] == 0).sum() / max((y[fit_sl] == 1).sum(), 1)); print("scale_pos_weight %.1f" % spw)

full data: (1852394, 17) fraud rate 0.00521
n=463,098  fit=333,430  val=37,048  test=92,620
fraud rate  fit 0.0059  val 0.0037  test 0.0039
tabular features: 18
scale_pos_weight 168.9


In [ ]:
# ---------- Cell 3: entities, prior-only degree features, richer prior-only aggregates ----------
ent = {"card": pd.factorize(df.cc_num)[0], "merchant": pd.factorize(df.merchant)[0], "category": pd.factorize(df.category)[0]}
print({k: int(v.max()) + 1 for k, v in ent.items()})

def prior_count(e):                                   
    return pd.Series(e).groupby(e).cumcount().values.astype(np.float32)
deg = np.column_stack([prior_count(ent[k]) for k in ["card", "merchant", "category"]])

# richer prior-only aggregates (use only earlier rows)
amt = df.amt.values.astype(np.float64)
def prior_mean(e):
    s = pd.Series(amt); g = s.groupby(e)
    cs = g.cumsum().values - amt; cn = g.cumcount().values
    return np.where(cn > 0, cs / np.maximum(cn, 1), np.nan)
card_pm = prior_mean(ent["card"]); merch_pm = prior_mean(ent["merchant"])
card_gap = (df.unix_time - df.groupby(ent["card"]).unix_time.shift(1)).values.astype(np.float64)
rich = np.column_stack([card_pm, merch_pm, card_gap, amt / np.where(card_pm > 0, card_pm, np.nan)]).astype(np.float32)
X_deg = np.hstack([X_tab, deg]); X_rich = np.hstack([X_tab, deg, rich])
print("with degrees:", X_deg.shape[1], " with rich aggregates:", X_rich.shape[1])

{'card': 996, 'merchant': 693, 'category': 14}
with degrees: 21  with rich aggregates: 25


In [4]:
# ---------- Cell 4: metrics and tree models (3 seeds each) ----------
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
import xgboost as xgb, lightgbm as lgb

def best_f1(yt, s):
    p, r, t = precision_recall_curve(yt, s); f = 2*p*r/np.maximum(p+r, 1e-12); return f[:-1].max(), t[f[:-1].argmax()]
def metrics(s_val, s_test):
    yv, yt = y[val_sl], y[test_sl]
    _, thr = best_f1(yv, s_val); pred = s_test >= thr
    tp = (pred & (yt == 1)).sum(); prec = tp / max(pred.sum(), 1); rec = tp / yt.sum()
    order = np.argsort(-s_test)
    return dict(AUPRC=average_precision_score(yt, s_test), AUC_ROC=roc_auc_score(yt, s_test),
                best_F1=best_f1(yt, s_test)[0], F1_at_val_thr=2*prec*rec/max(prec+rec, 1e-12),
                P_at_100=yt[order[:100]].mean(), P_at_500=yt[order[:500]].mean())

results, preds, models = [], {}, {}
def record(name, seed, s_val, s_test, extra=None):
    m = metrics(s_val, s_test); m.update(model=name, seed=seed, **(extra or {})); results.append(m)
    preds[(name, seed)] = s_test.astype(np.float32)
    print(f"{name:16s} seed {seed}  AUPRC {m['AUPRC']:.4f}  AUC {m['AUC_ROC']:.4f}  bestF1 {m['best_F1']:.4f}  P@100 {m['P_at_100']:.2f}")

def run_xgb(name, X, seed):
    t = time.time()
    m = xgb.XGBClassifier(n_estimators=CFG["MAX_TREES"], max_depth=6, learning_rate=CFG["TREE_LR"], subsample=0.8,
        colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="aucpr", early_stopping_rounds=CFG["ES_ROUNDS"],
        tree_method="hist", n_jobs=-1, random_state=seed)
    m.fit(X[fit_sl], y[fit_sl], eval_set=[(X[val_sl], y[val_sl])], verbose=False)
    record(name, seed, m.predict_proba(X[val_sl])[:, 1], m.predict_proba(X[test_sl])[:, 1],
           dict(best_iter=int(m.best_iteration), secs=round(time.time()-t)))
    models[(name, seed)] = m

def run_lgb(name, X, seed):
    t = time.time()
    m = lgb.LGBMClassifier(n_estimators=CFG["MAX_TREES"], learning_rate=CFG["TREE_LR"], max_depth=6, num_leaves=63,
        min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, scale_pos_weight=spw,
        random_state=seed, n_jobs=-1, verbose=-1)
    m.fit(X[fit_sl], y[fit_sl], eval_set=[(X[val_sl], y[val_sl])], eval_metric="average_precision",
          callbacks=[lgb.early_stopping(100, verbose=False)])
    record(name, seed, m.predict_proba(X[val_sl])[:, 1], m.predict_proba(X[test_sl])[:, 1],
           dict(best_iter=int(m.best_iteration_), secs=round(time.time()-t)))

# seed 42 first (all four tree models); extra seeds for the XGBoost models are in Cell 5b
run_xgb("xgb_tab", X_tab, 42); run_lgb("lgb_tab", X_tab, 42)
run_xgb("xgb_deg", X_deg, 42); run_xgb("xgb_rich", X_rich, 42)

xgb_tab          seed 42  AUPRC 0.8680  AUC 0.9949  bestF1 0.8331  P@100 1.00
lgb_tab          seed 42  AUPRC 0.6075  AUC 0.9863  bestF1 0.6281  P@100 0.99
xgb_deg          seed 42  AUPRC 0.8504  AUC 0.9949  bestF1 0.8052  P@100 1.00
xgb_rich         seed 42  AUPRC 0.8888  AUC 0.9964  bestF1 0.8471  P@100 1.00


In [ ]:
# ---------- Cell 5: heterogeneous GAT with and without neighbour messages, then hybrids  ----------
if CFG["RUN_GAT"]:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch_geometric.data import HeteroData
    from torch_geometric.nn import HeteroConv, GATConv
    torch.set_num_threads(2)
    print("torch", torch.__version__, "- building graph...", flush=True)

    Xg = StandardScaler().fit(X_tab[:split_idx]).transform(X_tab).astype(np.float32)
    tr = np.arange(n) < split_idx
    def build_graph(messages):
        d = HeteroData(); d["transaction"].x = torch.tensor(Xg)
        ar = np.arange(n)
        d["transaction", "self_loop", "transaction"].edge_index = torch.tensor(np.vstack([ar, ar]), dtype=torch.long)
        if messages:
            rels = {"card": "used_card", "merchant": "at_merchant"}
            for k, rel in rels.items():
                e = ent[k]; cnt = np.bincount(e[tr], minlength=e.max()+1).astype(np.float64)
                mean_amt = np.bincount(e[tr], weights=amt[tr], minlength=e.max()+1) / np.maximum(cnt, 1)
                d[k].x = torch.tensor(np.column_stack([np.log1p(mean_amt), np.log1p(cnt)]).astype(np.float32))
                d[k, "rev_" + rel, "transaction"].edge_index = torch.tensor(np.vstack([e, ar]), dtype=torch.long)
        return d

    class HeteroGAT(nn.Module):
        def __init__(self, etypes, hidden, heads, out, dropout=0.3):
            super().__init__()
            self.conv1 = HeteroConv({e: GATConv((-1, -1), hidden, heads=heads, add_self_loops=False) for e in etypes}, aggr="sum")
            self.conv2 = HeteroConv({e: GATConv((-1, -1), out, heads=1, add_self_loops=False) for e in etypes if e[2] == "transaction"}, aggr="sum")
            self.drop = nn.Dropout(dropout); self.lin = nn.Linear(out, 1)
        def forward(self, x, ei):
            h = self.conv1(x, ei); h = {k: self.drop(F.elu(v)) for k, v in h.items()}
            for k in x:
                if k not in h: h[k] = x[k]
            t = self.conv2(h, ei)["transaction"]
            return self.lin(t).squeeze(-1), t

    def train_gat(messages, seed):
        torch.manual_seed(seed); np.random.seed(seed)
        d = build_graph(messages); model = HeteroGAT(list(d.edge_index_dict.keys()), CFG["GAT_HIDDEN"], CFG["GAT_HEADS"], CFG["GAT_OUT"])
        with torch.no_grad(): model(d.x_dict, d.edge_index_dict)
        print(f"  [{'msg' if messages else 'nomsg'}] graph ready, training starts", flush=True)
        opt = torch.optim.Adam(model.parameters(), lr=CFG["GAT_LR"], weight_decay=CFG["GAT_WD"])
        yt = torch.tensor(y, dtype=torch.float32); pw = torch.tensor(CFG["GAT_POSW"])
        fit_idx = torch.arange(0, val_start); best, best_ep, best_state, t0, log = -1, -1, None, time.time(), []
        for ep in range(CFG["GAT_EPOCHS"]):
            model.train(); opt.zero_grad()
            logit, _ = model(d.x_dict, d.edge_index_dict)
            loss = F.binary_cross_entropy_with_logits(logit[fit_idx], yt[fit_idx], pos_weight=pw); loss.backward(); opt.step()
            if ep % CFG["GAT_EVAL_EVERY"] == 0 or ep == CFG["GAT_EPOCHS"] - 1:
                model.eval()
                with torch.no_grad(): lv, _ = model(d.x_dict, d.edge_index_dict)
                v = average_precision_score(y[val_sl], lv[val_start:split_idx].numpy()); log.append((ep, float(loss), float(v)))
                if v > best: best, best_ep, best_state = v, ep, {k: t.clone() for k, t in model.state_dict().items()}
                print(f"  [{'msg' if messages else 'nomsg'}] ep {ep:3d} loss {loss.item():.4f} val AUPRC {v:.4f}  {time.time()-t0:.0f}s", flush=True)
            if time.time() - t0 > CFG["GAT_MAX_MIN"] * 60: print("  time cap reached", flush=True); break
        model.load_state_dict(best_state); model.eval()
        with torch.no_grad(): lg, emb = model(d.x_dict, d.edge_index_dict)
        return torch.sigmoid(lg).numpy(), emb.numpy(), best_ep, log

    gat_logs, emb_store = {}, {}
    for seed in CFG["GAT_SEEDS"]:
        for messages, tag in [(False, "nomsg"), (True, "msg")]:
            p, emb, bep, log = train_gat(messages, seed); gat_logs[(tag, seed)] = log; emb_store[(tag, seed)] = emb
            record("gat_" + tag, seed, p[val_sl], p[test_sl], dict(best_epoch=bep))
            run_xgb("xgb_emb_" + tag, np.hstack([X_tab, emb]), seed)
else:
    print("RUN_GAT is off")

torch 2.8.0 - building graph...
  [nomsg] graph ready, training starts
  [nomsg] ep   0 loss 0.6202 val AUPRC 0.0260  1s
  [nomsg] ep  10 loss 0.2083 val AUPRC 0.0971  4s
  [nomsg] ep  20 loss 0.0869 val AUPRC 0.1148  7s
  [nomsg] ep  30 loss 0.0866 val AUPRC 0.1394  11s
  [nomsg] ep  40 loss 0.0783 val AUPRC 0.1810  14s
  [nomsg] ep  50 loss 0.0700 val AUPRC 0.2154  18s
  [nomsg] ep  60 loss 0.0671 val AUPRC 0.2335  21s
  [nomsg] ep  70 loss 0.0655 val AUPRC 0.2436  24s
  [nomsg] ep  80 loss 0.0624 val AUPRC 0.2501  28s
  [nomsg] ep  90 loss 0.0606 val AUPRC 0.2631  31s
  [nomsg] ep 100 loss 0.0590 val AUPRC 0.2695  34s
  [nomsg] ep 110 loss 0.0580 val AUPRC 0.2759  38s
  [nomsg] ep 120 loss 0.0574 val AUPRC 0.2802  41s
  [nomsg] ep 130 loss 0.0559 val AUPRC 0.2838  45s
  [nomsg] ep 140 loss 0.0554 val AUPRC 0.2889  48s
  [nomsg] ep 149 loss 0.0554 val AUPRC 0.2925  51s
gat_nomsg        seed 42  AUPRC 0.3159  AUC 0.8730  bestF1 0.4000  P@100 0.62
xgb_emb_nomsg    seed 42  AUPRC 0.8612

In [ ]:
# ---------- Cell 5b: extra seeds for the XGBoost models ----------
for seed in CFG["SEEDS"][1:]:
    run_xgb("xgb_tab", X_tab, seed); run_xgb("xgb_deg", X_deg, seed); run_xgb("xgb_rich", X_rich, seed)

xgb_tab          seed 43  AUPRC 0.8653  AUC 0.9963  bestF1 0.8286  P@100 1.00
xgb_deg          seed 43  AUPRC 0.8511  AUC 0.9945  bestF1 0.8125  P@100 1.00
xgb_rich         seed 43  AUPRC 0.8793  AUC 0.9965  bestF1 0.8318  P@100 1.00
xgb_tab          seed 44  AUPRC 0.8655  AUC 0.9956  bestF1 0.8271  P@100 1.00
xgb_deg          seed 44  AUPRC 0.8454  AUC 0.9947  bestF1 0.8125  P@100 1.00
xgb_rich         seed 44  AUPRC 0.8895  AUC 0.9967  bestF1 0.8468  P@100 1.00


In [7]:
# ---------- Cell 6: summary, paired differences, bootstrap, save ----------
res = pd.DataFrame(results); res.to_csv(f"{CFG['OUT_DIR']}/results_long.csv", index=False)
mcols = ["AUPRC", "AUC_ROC", "best_F1", "F1_at_val_thr", "P_at_100", "P_at_500"]
summ = res.groupby("model")[mcols].agg(["mean", "std"]).round(4); summ.to_csv(f"{CFG['OUT_DIR']}/summary_mean_sd.csv")
order = [m for m in ["xgb_tab","lgb_tab","xgb_deg","xgb_rich","xgb_emb_nomsg","xgb_emb_msg","gat_nomsg","gat_msg"] if m in set(res.model)]
print(summ.loc[order].to_string())

pairs = [("xgb_deg","xgb_tab"),("xgb_rich","xgb_tab"),("lgb_tab","xgb_tab"),("xgb_emb_nomsg","xgb_tab"),
         ("xgb_emb_msg","xgb_tab"),("xgb_emb_msg","xgb_emb_nomsg"),("gat_msg","gat_nomsg")]
rows, brows = [], []; rng = np.random.default_rng(42); yt = y[test_sl]
for a, b in pairs:
    seeds = sorted(set(res[res.model == a].seed) & set(res[res.model == b].seed))
    if not seeds: continue
    d = np.array([res[(res.model == a) & (res.seed == s)].AUPRC.iloc[0] - res[(res.model == b) & (res.seed == s)].AUPRC.iloc[0] for s in seeds])
    rows.append(dict(a=a, b=b, n_seeds=len(seeds), mean_delta=d.mean(), sd_delta=d.std(ddof=1) if len(d) > 1 else np.nan, seeds_a_better=int((d > 0).sum())))
    s0 = seeds[0]; pa, pb = preds[(a, s0)], preds[(b, s0)]; diffs = []
    for _ in range(CFG["N_BOOT"]):
        i = rng.integers(0, len(yt), len(yt)); diffs.append(average_precision_score(yt[i], pa[i]) - average_precision_score(yt[i], pb[i]))
    lo, hi = np.percentile(diffs, [2.5, 97.5]); brows.append(dict(a=a, b=b, seed=s0, mean=np.mean(diffs), lo=lo, hi=hi, excludes_zero=bool(lo > 0 or hi < 0)))
pd.DataFrame(rows).round(4).to_csv(f"{CFG['OUT_DIR']}/paired_deltas.csv", index=False)
pd.DataFrame(brows).round(4).to_csv(f"{CFG['OUT_DIR']}/paired_bootstrap.csv", index=False)
print(pd.DataFrame(rows).round(4).to_string()); print(pd.DataFrame(brows).round(4).to_string())
np.savez_compressed(f"{CFG['OUT_DIR']}/test_predictions.npz", y_test=yt, **{f"{k[0]}__{k[1]}": v for k, v in preds.items()})
json.dump(dict(python=sys.version, platform=platform.platform(), cpus=os.cpu_count(), cfg={k: v for k, v in CFG.items() if k != "DATA_DIR"},
               n=int(n), n_fit=int(val_start), n_val=int(split_idx - val_start), n_test=int(n - split_idx),
               fraud_rate=dict(fit=float(y[fit_sl].mean()), val=float(y[val_sl].mean()), test=float(y[test_sl].mean())),
               entities={k: int(v.max()) + 1 for k, v in ent.items()}),
          open(f"{CFG['OUT_DIR']}/run_info.json", "w"), indent=2)
print("saved to", CFG["OUT_DIR"])

                AUPRC         AUC_ROC         best_F1         F1_at_val_thr         P_at_100         P_at_500        
                 mean     std    mean     std    mean     std          mean     std     mean     std     mean     std
model                                                                                                                
xgb_tab        0.8663  0.0015  0.9956  0.0007  0.8296  0.0031        0.8126  0.0215   1.0000  0.0000   0.6167  0.0031
lgb_tab        0.6075     NaN  0.9863     NaN  0.6281     NaN        0.6253     NaN   0.9900     NaN   0.4980     NaN
xgb_deg        0.8490  0.0031  0.9947  0.0002  0.8101  0.0042        0.8034  0.0085   1.0000  0.0000   0.6047  0.0012
xgb_rich       0.8859  0.0057  0.9965  0.0002  0.8419  0.0087        0.8343  0.0054   1.0000  0.0000   0.6313  0.0064
xgb_emb_nomsg  0.8672  0.0062  0.9949  0.0002  0.8410  0.0034        0.8320  0.0057   0.9933  0.0058   0.6167  0.0023
xgb_emb_msg    0.8768  0.0053  0.9955  0.0001  0.8395  0

In [ ]:
# ---------- Cell 7: SHAP for the hybrid with message passing (seed 42) ----------
if CFG["RUN_GAT"] and ("xgb_emb_msg", 42) in models:
    import shap, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    names = tab_names + [f"GAT_emb_{i}" for i in range(CFG["GAT_OUT"])]
    Xh = np.hstack([X_tab, emb_store[("msg", 42)]])[test_sl]
    idx = np.random.default_rng(42).choice(len(Xh), min(CFG["SHAP_N"], len(Xh)), replace=False)
    sv = shap.TreeExplainer(models[("xgb_emb_msg", 42)]).shap_values(Xh[idx])
    mabs = np.abs(sv).mean(0); nt = X_tab.shape[1]
    top = np.argsort(-mabs)[:20]
    pd.DataFrame(dict(rank=range(1, 21), feature=[names[i] for i in top], group=["embedding" if i >= nt else "tabular" for i in top],
                      mean_abs_shap=mabs[top].round(4))).to_csv(f"{CFG['OUT_DIR']}/shap_top20.csv", index=False)
    tab_pf, emb_pf = mabs[:nt].mean(), mabs[nt:].mean(); tot = mabs[:nt].sum() + mabs[nt:].sum()
    print("per-feature |SHAP|: tabular %.4f  embedding %.4f" % (tab_pf, emb_pf))
    print("share of total: tabular %.1f%%  embedding %.1f%%" % (100*mabs[:nt].sum()/tot, 100*mabs[nt:].sum()/tot))
    print(pd.read_csv(f"{CFG['OUT_DIR']}/shap_top20.csv").to_string(index=False))
    shap.summary_plot(sv, Xh[idx], feature_names=names, max_display=20, show=False)
    plt.title("SHAP summary: XGBoost + GAT embeddings (message passing), Sparkov, seed 42"); plt.tight_layout()
    plt.savefig(f"{CFG['OUT_DIR']}/shap_summary_sparkov.png", dpi=160); plt.show()
    # GAT validation curves for the thesis figure
    json.dump({f"{k[0]}_{k[1]}": v for k, v in gat_logs.items()}, open(f"{CFG['OUT_DIR']}/gat_logs.json", "w"))
else:
    print("SHAP skipped (needs RUN_GAT on and a GAT seed 42)")

per-feature |SHAP|: tabular 0.3727  embedding 0.1974
share of total: tabular 51.5%  embedding 48.5%
 rank    feature     group  mean_abs_shap
    1        amt   tabular         1.8611
    2       hour   tabular         1.1344
    3   category   tabular         0.8970
    4    log_amt   tabular         0.7317
    5 GAT_emb_17 embedding         0.5678
    6 GAT_emb_10 embedding         0.4534
    7  GAT_emb_3 embedding         0.3468
    8   city_pop   tabular         0.3248
    9   merchant   tabular         0.3139
   10        age   tabular         0.3083
   11  GAT_emb_4 embedding         0.2545
   12 GAT_emb_20 embedding         0.2404
   13  GAT_emb_7 embedding         0.2324
   14 GAT_emb_28 embedding         0.2302
   15 GAT_emb_21 embedding         0.2288
   16  GAT_emb_5 embedding         0.2090
   17 GAT_emb_24 embedding         0.2067
   18 GAT_emb_23 embedding         0.2015
   19 GAT_emb_19 embedding         0.1989
   20  GAT_emb_6 embedding         0.1977
